> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, uploads the
> package and talks to the TCA9548A mux and the TMP102 on I2C bus 1 (read-only transfers, plus
> selecting and releasing mux channel 2). *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against `micropython_i2c.sim.bus.FakeBus`, writes the module and its tests, and needs no
>   board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

Notebook-only: import the package before anything else. Its `__init__` imports `connection`,
and the header cell below rewrites `connection.py`; importing the package first means no later
import reads the half-written file.

In [ ]:
import micropython_i2c  # noqa: F401

# Hardware and configuration

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`) |
| **Mux** | `FakeMux` at `0x71` (one level) and `0x70` -> `0x71` (two levels) | SparkFun Qwiic Mux (TCA9548A) at `0x70` |
| **Parts used** | `FakeRegisterPart` at `0x35` | TMP102 at `0x48` on mux channel 2: temperature and configuration registers read, never written |
| **Result** | not run yet | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): the connection layer moved from a hand-written file into this notebook.
Its public API and behaviour are unchanged: every driver depends on them.

# Python file
location: `src/micropython_i2c/connection.py`

tests: `tests/test_connection.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/connection.py
"""``I2CConnection``: one bound (bus, mux path, address) transport handle.

The PyVISA-resource analogue. A connection pins down exactly which physical device is
being talked to, and exposes the ``machine.I2C`` transfer methods with the address
argument removed, so a driver holding one cannot address the wrong device.

Mux paths are lists of ``[mux_addr, channel]`` hops, outermost first::

    []                        # device sits directly on the bus
    [[0x70, 3]]               # behind channel 3 of the mux at 0x70
    [[0x70, 3], [0x71, 0]]    # ...then channel 0 of the mux at 0x71

Every transfer takes the per-bus lock, replays the path, transfers and releases. The
path is re-selected on *every* transfer because mux state is global to the bus and any
other connection may have changed it; doing the selection inside the same critical
section as the transfer is what makes the pair atomic.

Generated by ``dev_notebooks/connection_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None.

## Internal
`default_lock`, for a connection made without the bus's shared lock.

In [ ]:
%%extract ../src/micropython_i2c/connection.py -a
from micropython_i2c.compat import default_lock

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_connection.py
import pytest

from micropython_i2c.compat import NullLock, default_lock
from micropython_i2c.connection import I2CConnection, path_release, path_select
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c.compat import NullLock, default_lock
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart

# Intent
**Purpose:** one object per physical device that carries everything needed to reach it: the
bus, the mux hops in front of it, its address and the bus lock. A driver holding a connection
calls `readfrom_mem(reg, n)` and never names an address or a mux.

**Why route on every transfer.** Mux state belongs to the bus, not to a device. Between two
transfers of one driver, another driver may have switched the mux to its own channel. So each
transfer, under the bus lock, first writes the whole path and then transfers:

```mermaid
sequenceDiagram
    participant D as driver
    participant C as I2CConnection
    participant B as machine.I2C
    D->>C: readfrom_mem(reg, n)
    C->>C: take bus lock
    C->>B: writeto(mux 0x70, 1 << ch)   (each hop, outermost first)
    C->>B: readfrom_mem(addr, reg, n)
    C->>C: release bus lock
    C-->>D: bytes
```

The path is **not** released after a transfer: the next transfer selects its own path anyway,
and leaving the mux alone halves the traffic. `path_release` exists for the manager and for
scans, which need everything downstream disconnected.

**Used by:** `node.I2CNode` (every device and mux talks through one), the manager.

**Not its job:** knowing what a device is (that is `node`), checking a path is valid (that is
`routing`).

## What the module defines

| Name | What it is |
|---|---|
| `path_select` | write a one-hot channel mask to each mux on a path, outermost first |
| `path_release` | deselect every mux on a path, innermost first |
| `I2CConnection` | locked, routed transport for one device: `machine.I2C` without the address |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/connection.py -a
__all__ = ["I2CConnection", "path_release", "path_select"]

## `path_select` / `path_release`
The two path operations, as plain functions of a bus so the manager and scans can use them
without a connection. Both expect the caller to hold the bus lock.

| Function | Order | Why that order |
|---|---|---|
| `path_select` | outermost first | a downstream mux is reachable only once its upstream mux has switched |
| `path_release` | innermost first | deselecting the outer mux first would cut the inner one off before it is told to let go |

Each mux gets a one-hot mask, `1 << channel`, which deselects every other channel on that mux.

In [ ]:
%%extract ../src/micropython_i2c/connection.py -a
def path_select(bus, path):
    """Write a one-hot channel mask to each mux on ``path``, outermost first.

    Each write must land before the next, since a downstream mux only becomes
    reachable once its upstream mux has switched. Call with the bus lock held.

    Args:
        bus: The ``machine.I2C`` object.
        path: Resolved hops as ``[[mux_addr, channel], ...]``.

    Raises:
        OSError: If a mux on the path does not acknowledge (absent, or unreachable).
    """
    for mux_addr, channel in path:
        bus.writeto(mux_addr, bytes([1 << channel]))


def path_release(bus, path):
    """Deselect every mux on ``path``, innermost first.

    Order matters: deselecting the outer mux first would cut the inner mux off the
    bus before it could be told to let go. Call with the bus lock held.

    Args:
        bus: The ``machine.I2C`` object.
        path: Resolved hops as ``[[mux_addr, channel], ...]``.

    Raises:
        OSError: If a mux on the path does not acknowledge.
    """
    for mux_addr, _ in reversed(path):
        bus.writeto(mux_addr, b"\x00")

### validation/test
A two-level chain for the cells below: mux `0x70`, a second mux `0x71` behind its channel 1,
and a register part at `0x35` behind channel 3 of the second mux. The tests use it too.

In [ ]:
%%extract ../tests/test_connection.py -a
CHAIN = [[0x70, 1], [0x71, 3]]  # the path to the part at 0x35


def chain_bus():
    """Mux 0x70 -> ch 1 -> mux 0x71 -> ch 3 -> register part at 0x35 (regs 0x0E, 0x0F)."""
    bus = FakeBus()
    bus.attach(0x70, FakeMux())
    bus.attach(0x71, FakeMux(), path=[(0x70, 1)])
    bus.attach(0x35, FakeRegisterPart({0x0E: 0x49, 0x0F: 0x54}), path=[(0x70, 1), (0x71, 3)])
    return bus

#### validation: nominal usage
Select the chain: the part appears in a scan and the bus log shows the outer mux written
first. Release it: the inner mux is written first and the part disappears.

In [ ]:
bus = chain_bus()
print("scan before:", [hex(a) for a in bus.scan()])
path_select(bus, CHAIN)
print("scan selected:", [hex(a) for a in bus.scan()], "| writes:", bus.log)
bus.log.clear()
path_release(bus, CHAIN)
print("scan released:", [hex(a) for a in bus.scan()], "| writes:", bus.log)

#### validation: anti-nominal usage
Releasing in the wrong order cuts the inner mux off: the second write finds nobody. This is
what `path_release` avoids by going innermost first.

In [ ]:
bus = chain_bus()
path_select(bus, CHAIN)
try:
    for mux_addr, _ in CHAIN:  # outermost first: the wrong order for a release
        bus.writeto(mux_addr, b"\x00")
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("releasing outermost first should have left 0x71 unreachable")

A path through a mux that is not there fails at the select, before any transfer.

In [ ]:
bus = chain_bus()
try:
    path_select(bus, [[0x74, 0]])  # no mux at 0x74
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("selecting through an absent mux should have raised OSError")

#### unit

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_path_select_writes_one_hot_masks_outermost_first():
    bus = chain_bus()
    path_select(bus, CHAIN)
    assert bus.log == [("w", 0x70, b"\x02"), ("w", 0x71, b"\x08")]
    assert 0x35 in bus.scan()

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_path_release_deselects_innermost_first():
    bus = chain_bus()
    path_select(bus, CHAIN)
    bus.log.clear()
    path_release(bus, CHAIN)
    assert bus.log == [("w", 0x71, b"\x00"), ("w", 0x70, b"\x00")]
    assert bus.scan() == [0x70]

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_empty_path_writes_nothing():
    bus = chain_bus()
    path_select(bus, [])
    path_release(bus, [])
    assert bus.log == []

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_path_select_through_absent_mux_raises_oserror():
    bus = chain_bus()
    with pytest.raises(OSError):
        path_select(bus, [[0x74, 0]])

## `I2CConnection`
The transport handle. It mirrors the `machine.I2C` transfer methods with the address
argument removed, in two groups:

| Group | Methods | For |
|---|---|---|
| register | `readfrom_mem`, `readfrom_mem_into`, `writeto_mem` | parts with a register map; `addrsize` is the register-number width in bits (8, or 16 for the VL53L4CD) |
| raw | `writeto`, `readfrom`, `readfrom_into` | command-driven parts and muxes |

Each method takes the lock, selects the path, transfers, and lets the lock go, also when the
transfer raises. `path` defaults to `[]` (a part on the bare bus). Without a `lock` the
connection makes a private one, which is only correct when nothing else uses the bus; the
manager always passes the bus's shared lock.

In [ ]:
%%extract ../src/micropython_i2c/connection.py -a
class I2CConnection:
    """Locked, routed transport for one device. Mirrors ``machine.I2C`` minus ``addr``.

    Args:
        bus: The underlying ``machine.I2C``. Not hidden; ``conn.bus`` stays usable.
        address: 7-bit device address, fixed for the life of the connection.
        path: Resolved mux hops, outermost first. Empty or None for a bare-bus device.
        lock: The per-bus mutex, shared by everything on the bus. If omitted a
            private lock is created, which is only correct when this connection is the
            sole user of the bus.

    Attributes:
        bus, addr, path, lock: The binding. Not meant to be mutated; build a new
            connection instead.
    """

    def __init__(self, bus, address, path=None, lock=None):
        self.bus = bus
        self.addr = address
        self.path = path or []
        self.lock = lock or default_lock()

    def __repr__(self):
        hops = "".join(" via {:#04x}:{}".format(a, c) for a, c in self.path)
        return "<I2CConnection {:#04x}{}>".format(self.addr, hops)

    def path_select(self):
        """Select this connection's mux path. Call with ``self.lock`` held."""
        path_select(self.bus, self.path)

    # Register-oriented transfers (write register number, repeated start, read).
    # ``addrsize`` is the register-number width in bits, as in machine.I2C: 8 for most
    # parts, 16 for parts such as the VL53L4CD.

    def readfrom_mem(self, reg, nbytes, addrsize=8):
        """Read ``nbytes`` from register ``reg``."""
        with self.lock:
            path_select(self.bus, self.path)
            return self.bus.readfrom_mem(self.addr, reg, nbytes, addrsize=addrsize)

    def readfrom_mem_into(self, reg, buf, addrsize=8):
        """Read register ``reg`` into ``buf``; avoids an allocation in hot loops."""
        with self.lock:
            path_select(self.bus, self.path)
            self.bus.readfrom_mem_into(self.addr, reg, buf, addrsize=addrsize)

    def writeto_mem(self, reg, data, addrsize=8):
        """Write ``data`` to register ``reg``."""
        with self.lock:
            path_select(self.bus, self.path)
            self.bus.writeto_mem(self.addr, reg, data, addrsize=addrsize)

    # Raw transfers, for command-driven parts with no register map.

    def writeto(self, data):
        """Write raw bytes. Returns the ACK count from the bus."""
        with self.lock:
            path_select(self.bus, self.path)
            return self.bus.writeto(self.addr, data)

    def readfrom(self, nbytes):
        """Read ``nbytes`` of raw data."""
        with self.lock:
            path_select(self.bus, self.path)
            return self.bus.readfrom(self.addr, nbytes)

    def readfrom_into(self, buf):
        """Read raw data into ``buf``."""
        with self.lock:
            path_select(self.bus, self.path)
            self.bus.readfrom_into(self.addr, buf)

### validation/test: transfers and routing
A one-level bus for the cells below, the one the routing tests used before this notebook
existed: mux `0x71`, register part at `0x35` behind its channel 3.

In [ ]:
%%extract ../tests/test_connection.py -a
def one_mux_bus():
    """Mux 0x71 -> ch 3 -> register part at 0x35 (regs 0x0D..0x0F)."""
    bus = FakeBus()
    bus.attach(0x71, FakeMux())
    bus.attach(0x35, FakeRegisterPart({0x0E: 0x49, 0x0F: 0x54, 0x0D: 0x01}), path=[(0x71, 3)])
    return bus

#### validation: nominal usage
A connection through the mux reads the part even though the mux starts deselected; each
transfer writes the mux first. Then every other transfer method, and the `repr`.

In [ ]:
bus = one_mux_bus()
conn = I2CConnection(bus, 0x35, [[0x71, 3]], NullLock())
print(conn)
print("readfrom_mem(0x0E, 2):", conn.readfrom_mem(0x0E, 2), "| log:", bus.log[-2:])
buf = bytearray(3)
conn.readfrom_mem_into(0x0D, buf)
print("readfrom_mem_into(0x0D):", bytes(buf))
conn.writeto_mem(0x20, b"\xaa\xbb")
print("after writeto_mem(0x20):", conn.readfrom_mem(0x20, 2))
print("writeto returns the byte count on FakeBus:", conn.writeto(b"\x0e"))
print("readfrom(2):", conn.readfrom(2))
raw = bytearray(2)
conn.readfrom_into(raw)
print("readfrom_into:", bytes(raw))
print("bare-bus repr:", I2CConnection(bus, 0x71))

#### validation: anti-nominal usage
The bare bus cannot reach the part while the mux is deselected; this is why a connection
selects its path itself.

In [ ]:
bus = one_mux_bus()
try:
    bus.readfrom_mem(0x35, 0x0E, 1)  # mux 0x71 still deselected
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("the part should be unreachable before the mux is switched")

A connection to an address nobody answers raises the bus's `OSError` unchanged.

In [ ]:
try:
    I2CConnection(bus, 0x36, [[0x71, 3]]).readfrom_mem(0x0E, 1)  # nothing at 0x36
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("reading an absent part should have raised OSError")

#### unit

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_connection_selects_path_each_transfer():
    bus = one_mux_bus()
    conn = I2CConnection(bus, 0x35, [[0x71, 3]], NullLock())
    with pytest.raises(OSError):
        bus.readfrom_mem(0x35, 0x0E, 1)  # unreachable until the mux is switched
    assert conn.readfrom_mem(0x0E, 2) == b"\x49\x54"
    assert bus.log[-2] == ("w", 0x71, b"\x08")

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_path_reselected_after_another_connection_switched_the_mux():
    bus = one_mux_bus()
    conn = I2CConnection(bus, 0x35, [[0x71, 3]], NullLock())
    conn.readfrom_mem(0x0E, 1)
    bus.writeto(0x71, b"\x01")  # someone else routes the mux to channel 0
    assert conn.readfrom_mem(0x0F, 1) == b"\x54"

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_every_transfer_method_routes_and_drops_the_address():
    bus = one_mux_bus()
    conn = I2CConnection(bus, 0x35, [[0x71, 3]], NullLock())
    conn.writeto_mem(0x20, b"\xaa\xbb")
    assert conn.readfrom_mem(0x20, 2) == b"\xaa\xbb"
    buf = bytearray(2)
    conn.readfrom_mem_into(0x0E, buf)
    assert bytes(buf) == b"\x49\x54"
    assert conn.writeto(b"\x01\x02") == 2
    assert conn.readfrom(1) == b"\x00"  # FakeRegisterPart raw reads start at register 0
    raw = bytearray(1)
    conn.readfrom_into(raw)
    assert bytes(raw) == b"\x00"
    transfers = [e for e in bus.log if e[1] == 0x35]
    selects = [e for e in bus.log if e == ("w", 0x71, b"\x08")]
    assert len(selects) == len(transfers) == 6

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_two_level_path_routes_through_both_muxes():
    bus = chain_bus()
    conn = I2CConnection(bus, 0x35, CHAIN, NullLock())
    assert conn.readfrom_mem(0x0E, 2) == b"\x49\x54"
    assert bus.log[:2] == [("w", 0x70, b"\x02"), ("w", 0x71, b"\x08")]

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_absent_part_raises_oserror():
    bus = one_mux_bus()
    with pytest.raises(OSError):
        I2CConnection(bus, 0x36, [[0x71, 3]]).readfrom_mem(0x0E, 1)

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_defaults_and_repr():
    bus = FakeBus()
    bare = I2CConnection(bus, 0x48)
    assert bare.path == [] and bare.addr == 0x48 and bare.bus is bus
    assert repr(bare) == "<I2CConnection 0x48>"
    routed = I2CConnection(bus, 0x35, [[0x70, 1], [0x71, 3]])
    assert repr(routed) == "<I2CConnection 0x35 via 0x70:1 via 0x71:3>"

### validation/test: locking
A bus wrapper that notes, at each transfer, whether the lock was held. The path writes and the
transfer must all happen with the lock held, and the lock must be free again afterwards, also
when the transfer failed.

In [ ]:
%%extract ../tests/test_connection.py -a
class LockWatchBus:
    """Wraps a bus; ``held`` records, per transfer, whether ``lock`` was held at that moment."""

    def __init__(self, inner, lock):
        self.inner = inner
        self.lock = lock
        self.held = []

    def _note(self):
        free = self.lock.acquire(False)
        if free:
            self.lock.release()
        self.held.append(not free)

    def writeto(self, addr, data, stop=True):
        self._note()
        return self.inner.writeto(addr, data, stop)

    def readfrom(self, addr, nbytes, stop=True):
        self._note()
        return self.inner.readfrom(addr, nbytes, stop)

    def readfrom_into(self, addr, buf, stop=True):
        self._note()
        self.inner.readfrom_into(addr, buf, stop)

    def writeto_mem(self, addr, reg, data, addrsize=8):
        self._note()
        self.inner.writeto_mem(addr, reg, data, addrsize)

    def readfrom_mem(self, addr, reg, nbytes, addrsize=8):
        self._note()
        return self.inner.readfrom_mem(addr, reg, nbytes, addrsize)

    def readfrom_mem_into(self, addr, reg, buf, addrsize=8):
        self._note()
        self.inner.readfrom_mem_into(addr, reg, buf, addrsize)

#### validation: nominal usage

In [ ]:
lock = default_lock()
watch = LockWatchBus(one_mux_bus(), lock)
conn = I2CConnection(watch, 0x35, [[0x71, 3]], lock)
conn.readfrom_mem(0x0E, 2)
print("lock held at each bus call (select, transfer):", watch.held)
print("lock free afterwards:", lock.acquire(False))
lock.release()
print("no lock given -> private lock, not shared:", I2CConnection(watch, 0x35).lock is not lock)

#### validation: anti-nominal usage
A failed transfer still lets the lock go.

In [ ]:
try:
    conn_absent = I2CConnection(watch, 0x36, [[0x71, 3]], lock)
    conn_absent.readfrom(1)  # nothing at 0x36
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("reading an absent part should have raised OSError")
finally:
    free = lock.acquire(False)
    print("lock free after the failure:", free)
    if free:
        lock.release()

#### unit

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_select_and_transfer_happen_under_the_lock():
    lock = default_lock()
    watch = LockWatchBus(one_mux_bus(), lock)
    conn = I2CConnection(watch, 0x35, [[0x71, 3]], lock)
    conn.readfrom_mem(0x0E, 2)
    conn.writeto(b"\x00")
    assert watch.held == [True, True, True, True]
    assert lock.acquire(False)
    lock.release()

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_lock_released_when_transfer_fails():
    lock = default_lock()
    conn = I2CConnection(LockWatchBus(one_mux_bus(), lock), 0x36, [[0x71, 3]], lock)
    with pytest.raises(OSError):
        conn.readfrom_mem(0x0E, 1)
    assert lock.acquire(False)
    lock.release()

In [ ]:
%%extract ../tests/test_connection.py -a --strip-ipytest
%%ipytest


def test_shared_lock_is_used_and_default_lock_is_private():
    bus, shared = FakeBus(), NullLock()
    assert I2CConnection(bus, 0x48, lock=shared).lock is shared
    a, b = I2CConnection(bus, 0x48), I2CConnection(bus, 0x49)
    assert a.lock is not b.lock

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, and on I2C bus 1 selects and releases channel 2 of the TCA9548A at `0x70` and reads
> the TMP102 at `0x48` (temperature and configuration registers). Nothing is written to the
> TMP102.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

**Hardware and configuration** (same as the table at the top):

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`) |
| **Mux** | `FakeMux` at `0x71` (one level) and `0x70` -> `0x71` (two levels) | SparkFun Qwiic Mux (TCA9548A) at `0x70` |
| **Parts used** | `FakeRegisterPart` at `0x35` | TMP102 at `0x48` on mux channel 2: temperature and configuration registers read, never written |
| **Result** | not run yet | not run yet |

Layout: connect and set up, the **matching tests** (one per unit test above that makes sense
on the board, same name), then the tally and disconnect. The two-level chain has no bench
counterpart (one mux on the bench), so its tests have no match here.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.boards import bus_open
from micropython_i2c.compat import NullLock, default_lock
from micropython_i2c.connection import I2CConnection, path_release, path_select
from micropython_i2c.debug import check, check_raises, check_reset, check_summary, info
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)

check_reset()

Notebook-only: the bus and the bench constants, then release the mux so every check starts
from a deselected mux. If the bench is rewired, change the constants.

**What to do:** check the scan shows `0x70` (the mux). `0x48` must not show: it is behind the
mux.

In [ ]:
BUS_ID, MUX_ADDR, TMP_ADDR = 1, 0x70, 0x48
TMP_PATH = [[MUX_ADDR, 2]]
TMP_REG_TEMP, TMP_REG_CONFIG = 0x00, 0x01

bus = bus_open("feathers3", BUS_ID, freq=100_000)
bus_lock = default_lock()
path_release(bus, TMP_PATH)
print("scan, mux released:", [hex(a) for a in bus.scan()])

## Matching tests

**Matches** `test_path_select_writes_one_hot_masks_outermost_first` and
`test_path_release_deselects_innermost_first`, on one level: selecting channel 2 writes
`0x04` to the mux and makes the TMP102 answer a scan; releasing writes `0x00` and hides it.

**What to do:** nothing.

In [ ]:
path_select(bus, TMP_PATH)
check("select: mux control reads 0x04", bus.readfrom(MUX_ADDR, 1) == b"\x04")
check("select: 0x48 answers a scan", TMP_ADDR in bus.scan())
path_release(bus, TMP_PATH)
check("release: mux control reads 0x00", bus.readfrom(MUX_ADDR, 1) == b"\x00")
check("release: 0x48 gone from the scan", TMP_ADDR not in bus.scan())

**Matches** `test_empty_path_writes_nothing`: an empty path leaves the mux as it was.

**What to do:** nothing.

In [ ]:
path_select(bus, TMP_PATH)
path_select(bus, [])
path_release(bus, [])
check("test_empty_path_writes_nothing", bus.readfrom(MUX_ADDR, 1) == b"\x04")
path_release(bus, TMP_PATH)

**Matches** `test_path_select_through_absent_mux_raises_oserror`: nothing answers at `0x74`
on this bench.

**What to do:** nothing. If `0x74` showed in the scan above, skip this cell.

In [ ]:
check_raises("test_path_select_through_absent_mux_raises_oserror", OSError,
             path_select, bus, [[0x74, 0]])  # fmt: skip

**Matches** `test_connection_selects_path_each_transfer`: with the mux released the bare bus
cannot reach the TMP102; the connection reaches it, and leaves the mux on channel 2. The
configuration register's resolution bits (`R1 R0`, bits 14..13) always read `11`.

**What to do:** nothing.

In [ ]:
conn = I2CConnection(bus, TMP_ADDR, TMP_PATH, bus_lock)
print(conn)
check_raises("bare bus cannot reach 0x48", OSError, bus.readfrom_mem, TMP_ADDR, TMP_REG_CONFIG, 2)
cfg = conn.readfrom_mem(TMP_REG_CONFIG, 2)
check("test_connection_selects_path_each_transfer", cfg[0] & 0x60 == 0x60,
      "config {:#06x}".format(int.from_bytes(cfg, "big")))  # fmt: skip
check("mux left on channel 2", bus.readfrom(MUX_ADDR, 1) == b"\x04")

**Matches** `test_path_reselected_after_another_connection_switched_the_mux`: release the mux
behind the connection's back; the next transfer selects it again.

**What to do:** nothing.

In [ ]:
path_release(bus, TMP_PATH)
again = conn.readfrom_mem(TMP_REG_CONFIG, 2)
check("test_path_reselected_after_another_connection_switched_the_mux", again[0] & 0x60 == 0x60)

**Matches** `test_every_transfer_method_routes_and_drops_the_address`, read-only half: the
register reads on the TMP102, the raw transfers on the mux's own connection (writing the mux
control byte is the select and release this notebook allows). The temperature must be a room
reading. `writeto` returns the ACK count on MicroPython: 1 for one byte.

**What to do:** nothing. If the room is outside 5..45 C, read the printed value instead.

In [ ]:
path_release(bus, TMP_PATH)
buf = bytearray(2)
conn.readfrom_mem_into(TMP_REG_CONFIG, buf)
check("readfrom_mem_into matches readfrom_mem", bytes(buf) == bytes(cfg))
raw_t = int.from_bytes(conn.readfrom_mem(TMP_REG_TEMP, 2), "big")
extended = raw_t & 1  # EM mode: 13-bit result, flag in bit 0
counts = raw_t >> (3 if extended else 4)
bits = 13 if extended else 12
if counts & (1 << (bits - 1)):
    counts -= 1 << bits
temp_c = counts * 0.0625
check("temperature is a room reading", 5 <= temp_c <= 45, "{:.2f} C".format(temp_c))

mux_conn = I2CConnection(bus, MUX_ADDR, [], bus_lock)
check("writeto returns the ACK count", mux_conn.writeto(b"\x04") == 1)
check("readfrom reads the control byte", mux_conn.readfrom(1) == b"\x04")
ctl = bytearray(1)
mux_conn.readfrom_into(ctl)
check("readfrom_into reads the control byte", ctl[0] == 0x04)
mux_conn.writeto(b"\x00")

**Matches** `test_absent_part_raises_oserror`: the TMP102 with the mux path left out is an
absent part.

**What to do:** nothing.

In [ ]:
path_release(bus, TMP_PATH)
check_raises("test_absent_part_raises_oserror", OSError,
             I2CConnection(bus, TMP_ADDR, [], bus_lock).readfrom_mem, TMP_REG_CONFIG, 1)  # fmt: skip

**Matches** `test_defaults_and_repr`.

**What to do:** nothing.

In [ ]:
bare = I2CConnection(bus, TMP_ADDR)
check("test_defaults_and_repr", bare.path == [] and repr(bare) == "<I2CConnection 0x48>"
      and repr(conn) == "<I2CConnection 0x48 via 0x70:2>")  # fmt: skip

**Matches** `test_select_and_transfer_happen_under_the_lock` and
`test_lock_released_when_transfer_fails`, on the board's own lock. A small wrapper notes, at
each bus call, whether the lock was held. On a build without `_thread` the lock is a
`NullLock` and there is nothing to check.

**What to do:** nothing.

In [ ]:
class LockWatchBus:
    """Wraps the bus; ``held`` records, per call, whether ``lock`` was held."""

    def __init__(self, inner, lock):
        self.inner = inner
        self.lock = lock
        self.held = []

    def _note(self):
        free = self.lock.acquire(0)
        if free:
            self.lock.release()
        self.held.append(not free)

    def writeto(self, addr, data, *args):
        self._note()
        return self.inner.writeto(addr, data, *args)

    def readfrom_mem(self, addr, reg, nbytes, addrsize=8):
        self._note()
        return self.inner.readfrom_mem(addr, reg, nbytes, addrsize=addrsize)


if isinstance(bus_lock, NullLock):
    info("no _thread on this build: the lock is a NullLock, nothing to check")
else:
    watch = LockWatchBus(bus, bus_lock)
    I2CConnection(watch, TMP_ADDR, TMP_PATH, bus_lock).readfrom_mem(TMP_REG_CONFIG, 2)
    check("test_select_and_transfer_happen_under_the_lock", watch.held == [True, True])
    path_release(bus, TMP_PATH)
    try:
        I2CConnection(watch, TMP_ADDR, [], bus_lock).readfrom_mem(TMP_REG_CONFIG, 1)
    except OSError:
        pass
    free = bus_lock.acquire(0)
    if free:
        bus_lock.release()
    check("test_lock_released_when_transfer_fails", free)

**Matches** `test_shared_lock_is_used_and_default_lock_is_private`.

**What to do:** nothing.

In [ ]:
check("test_shared_lock_is_used_and_default_lock_is_private",
      conn.lock is bus_lock and I2CConnection(bus, TMP_ADDR).lock is not bus_lock)  # fmt: skip

## Tally and disconnect

Notebook-only: leave the mux deselected, then the PASS/FAIL tally of every check in this
section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
path_release(bus, TMP_PATH)
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))